In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
bronzeOrder = spark.table(
"dev_sales.bronze.orders"
)

display(bronzeOrder)
#from pyspark.sql.window import Window
#from pyspark.sql.functions import col, row_number

window = Window.partitionBy(
"OrderID"
).orderBy(
col("LastModifiedDate").desc()
)


silverOrder = bronzeOrder \
.withColumn(
"rn",
row_number().over(window)
)\
.filter("rn=1")\
.drop("rn")
# To remove duplicates
# silverOrder = (
# bronzeOrder
# .dropDuplicates(["OrderID"])
# .withColumn(
# "LoadDate",
# current_timestamp()
# )
# )

silverOrder.write \
.format("delta")\
.mode("overwrite")\
.saveAsTable(
"dev_sales.silver.orders"
)
display(silverOrder)